# CMG STARS `.out` checker
For each `case<N>.out` file in a folder (other `.out` files are ignored): did the run finish normally, and what is the cumulative injection vs. production mismatch (%)?

`mismatch % = |cum_inj − cum_prod| / cum_inj × 100`, from the last `GROUP "FIELD" SUMMARY` block.

In [1]:
# ---- settings ----
FOLDER = "sim_gen_000"                  # folder containing case*.out
OUTPUT = "sim_gen_000/0_out_summary.csv"
BASIS     = "surface"          # "surface" (STC) or "reservoir"
RECURSIVE = False              # True to also search subfolders
MISMATCH_THRESHOLD = 5        # flag cases with mismatch % above this
TARGET_END_DATE = "2060-01-01"   # flag cases that stop before this date (None to disable)

In [2]:
import re
from pathlib import Path
import pandas as pd


NUM = r"[-+]?\d*\.?\d+(?:[EeDd][-+]?\d+)?"
FIELD_HDR = re.compile(r'S T A R S\s+G R O U P\s+"FIELD"\s+S U M M A R Y')
NORMAL_STOP = re.compile(r"Stopping end time reached|Stopping (?:on|at) .*?(?:time|date)|"
                         r"\bnormal(?:ly)? terminat", re.I)     # \b so "Abnormal" doesn't match
FAILED = re.compile(r"Abnormal Termination|Terminating simulation|FATAL ERROR", re.I)
END_OF_RUN = re.compile(r"Date and Time of End of Run")
SIM_DATE = re.compile(r"DATE:\s*(\d{4}):([A-Za-z]{3}):(\d{1,2})")
MONTHS = {m: i for i, m in enumerate(
    ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"], 1)}
CASE_RE = re.compile(r"case(\d+)\.out", re.I)   # only case<N>.out files
ERR_COUNT = re.compile(r"(\d+)\s+Warning messages\.\s+(\d+)\s+Error messages\.")


def to_float(s):
    return float(s.replace("D", "E").replace("d", "e"))


def last_field_block(lines):
    """Return lines of the last FIELD group summary block."""
    idx = [i for i, l in enumerate(lines) if FIELD_HDR.search(l)]
    if not idx:
        return None
    start = idx[-1]
    # block ends at the next '****' banner after the '====' closing line
    for j in range(start + 3, min(start + 200, len(lines))):
        if lines[j].lstrip().startswith("=====") and j > start + 20:
            return lines[start:j + 1]
    return lines[start:start + 120]


def total_col_value(line):
    """Last numeric token on a line = 'TOTAL FOR GROUP' column."""
    toks = re.findall(NUM, line.split("+", 1)[1] if "+" in line else line)
    return to_float(toks[-1]) if toks else None


def cum_from_block(block, basis="surface"):
    """Return (cum_inj, cum_prod) water from the FIELD block."""
    if basis == "surface":
        prod_hdr, inj_hdr = "surface cumulative production", "surface cumulative injection"
    else:
        prod_hdr, inj_hdr = "reservoir cumulative production", "reservoir cumulative injection"

    prod = inj = None
    section = None
    for l in block:
        # section headers start with exactly one space; data rows are indented more
        if re.match(r"^ \S", l):
            low = l.lower()
            section = "prod" if prod_hdr in low else "inj" if inj_hdr in low else None
            continue
        if section and re.match(r"^\s*Water\s", l):
            v = total_col_value(l)
            if section == "prod" and prod is None:
                prod = v
            elif section == "inj" and inj is None:
                inj = v
    return inj, prod


def sim_end_date(block):
    """Simulation date (YYYY-MM-DD) of the last FIELD summary = last time step written."""
    m = SIM_DATE.search(block[0]) if block else None
    if not m:
        return None
    y, mon, d = m.groups()
    mon = MONTHS.get(mon.upper())
    return f"{y}-{mon:02d}-{int(d):02d}" if mon else f"{y}:{m.group(2)}:{d}"


def analyze(path, basis="surface"):
    text = Path(path).read_text(errors="replace")
    lines = text.splitlines()

    normal_stop = bool(NORMAL_STOP.search(text))
    ended = bool(END_OF_RUN.search(text))
    errs = [int(m.group(2)) for m in ERR_COUNT.finditer(text)]
    has_errors = any(e > 0 for e in errs)
    failed = bool(FAILED.search(text))
    finished = normal_stop and ended and not has_errors and not failed

    mismatch = None
    block = last_field_block(lines)
    end_date = sim_end_date(block)
    if block:
        inj, prod = cum_from_block(block, basis)
        if inj and prod is not None:
            mismatch = abs(inj - prod) / abs(inj) * 100.0
    return finished, mismatch, end_date


def case_key(p):
    return int(CASE_RE.fullmatch(p.name).group(1))


def find_case_files(folder, recursive=False):
    """case<N>.out files only, sorted numerically (other .out files are ignored)."""
    pattern = "**/*.out" if recursive else "*.out"
    files = [p for p in Path(folder).glob(pattern) if CASE_RE.fullmatch(p.name)]
    return sorted(files, key=case_key)


def flag_cases(rows, threshold, target_end=None):
    """Return flagged rows (with a 'reason' field) for cases that
    - did not finish normally,
    - have mismatch > threshold (or no mismatch value),
    - did not reach target_end (YYYY-MM-DD; None disables this check)."""
    from datetime import date
    target = date.fromisoformat(target_end) if target_end else None
    bad = []
    for r in rows:
        reasons = []
        mm = r["mismatch_pct"]
        if r["finished_normally"] == "n":
            reasons.append("not finished normally")
        if mm is None:
            reasons.append("no mismatch value")
        elif mm > threshold:
            reasons.append(f"mismatch {mm:.2f}% > {threshold}%")
        if target:
            try:
                reached = date.fromisoformat(r["end_date"])
            except (TypeError, ValueError):
                reached = None
            if reached is None or reached < target:
                reasons.append(f"end date {r['end_date'] or 'NA'} < {target_end}")
        if reasons:
            bad.append({**r, "reason": "; ".join(reasons)})
    return bad

In [3]:
files = find_case_files(FOLDER, RECURSIVE)
if not files:
    raise FileNotFoundError(f"No case<N>.out files found in {Path(FOLDER).resolve()}")

rows = []
for p in files:
    try:
        ok, mm, end = analyze(p, BASIS)
    except Exception as e:
        print(f"{p.name}: failed ({e})")
        ok, mm, end = False, None, None
    rows.append({"case": p.stem, "finished_normally": "y" if ok else "n",
                 "mismatch_pct": None if mm is None else round(mm, 4),
                 "end_date": end})

df = pd.DataFrame(rows)
df.to_csv(OUTPUT, index=False)
print(f"Saved {Path(OUTPUT).resolve()} ({len(df)} cases)")
df

Saved \\smb-s-ees.oak.stanford.edu\groups\jcaers\jding\project_OMV_geothermal\well_placement_optimization_GA\ga_simulations\sim_gen_000\0_out_summary.csv (200 cases)


,case,finished_normally,mismatch_pct,end_date
0,case1,y,0.0149,2060-01-01
1,case2,y,0.2668,2060-01-01
2,case3,y,0.0198,2060-01-01
3,case4,y,1.3547,2060-01-01
4,case5,y,0.0231,2060-01-01
...,...,...,...,...
195,case196,y,0.5246,2060-01-01
196,case197,y,0.5964,2060-01-01
197,case198,y,0.3712,2060-01-01
198,case199,y,0.0064,2060-01-01


In [4]:
# ---- flagged cases: not finished normally, mismatch > threshold, or end date not reached ----
bad = flag_cases(rows, MISMATCH_THRESHOLD, TARGET_END_DATE)
print(f"{len(bad)} of {len(df)} cases flagged")
pd.DataFrame(bad, columns=["case", "finished_normally", "mismatch_pct", "end_date", "reason"])

1 of 200 cases flagged


,case,finished_normally,mismatch_pct,end_date,reason
0,case126,n,0.815,2050-01-01,not finished normally; end date 2050-01-01 < 2...
